# 13 · Vision ingestion — page images on top of transcription

Vision is **additive**: every agent prompt still contains the full `doc_text`.
Page images are appended only for vision-capable models, bounded by
`vision.max_pages` (0 = all pages). This notebook renders a real one-page PDF
inside the sandbox and shows the data-URIs `llm.vision.render_pdf_pages`
produces — no LLM call.

**What you'll see:** live `vision:` config, `pipeline_uses_vision()`, a
reportlab PDF rasterized by PyMuPDF, and the guarantee that `doc_text` is
never dropped when images are attached.

**Honesty label:** REAL render path (`llm.vision`, PyMuPDF). No model is
invoked. The multimodal `_build_multimodal` assembly is shown by inspecting
helpers, not by spending tokens. OFFLINE.


## Setup


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

import pipeline_lab as lab
lab.quiet_logs()
from llm import vision as v


## Live vision config (taxonomy.yaml + env overrides)


In [2]:
print("vision_enabled:       ", v.vision_enabled())
print("max_pages (0=all):    ", v.max_pages())
print("pipeline_uses_vision: ", v.pipeline_uses_vision())
print("qwen capable:         ", v.is_vision_capable("qwen/qwen3.7-flash"))
print("unknown model:        ", v.is_vision_capable("some-text-only-model"))


vision_enabled:        True
max_pages (0=all):     10
pipeline_uses_vision:  True
qwen capable:          True
unknown model:         False


## Render a one-page PDF

`write_lab_pdf` drops a real PDF in the sandbox inbox. `render_pdf_pages`
returns `data:image/png;base64,...` URIs — the same payload the sorter /
specialist prompts append as `image_url` parts.


In [3]:
env = lab.open_sandbox()
pdf = lab.write_lab_pdf(env["base_dir"], lab.DOC_CONTRACT, filename="vision_msa.pdf")
print("pdf:", pdf.name, "bytes:", pdf.stat().st_size)
pages = v.render_pdf_pages(pdf, cap=2, dpi=72)
print("pages rendered:", len(pages))
for i, uri in enumerate(pages, 1):
    head, _, rest = uri.partition(",")
    print(f"  page {i}: {head}  payload_chars={len(rest)}  prefix={uri[:48]}…")


pdf: vision_msa.pdf bytes: 1715


pages rendered: 1
  page 1: data:image/png;base64  payload_chars=32708  prefix=data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAm…


## Additive, never subtractive

A pipeline run on this PDF still stores transcription in `doc_text`. Images
ride alongside; a page cap never drops document text. (`cap<=0` renders every
page — the config `max_pages` only bounds the *image budget*.)


In [4]:
lab.script_all_specialists(env["client"])
run = lab.run_document(
    env, lab.DOC_CONTRACT, filename="vision_text_twin.txt",
    classification=lab.CLASSIFY_CONTRACT_HIGH, extraction=lab.EXTRACT_HIGH,
)
print("text twin path:", " → ".join(lab.path_of(run["steps"])))
print("stage:", run["final"].get("stage"))
print("doc_text present:", bool(run["final"].get("doc_text")))
lab.close_sandbox(env)


text twin path: intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
stage: archived
doc_text present: True


## Where to go next

- **00 pipeline_anatomy** — which agents are vision-capable
- **11 huggingface_corpora** — `mailroom-cuad-contracts` is the image-folder CUAD surface
- **01 happy_path_run** — the text path this notebook sits on top of
